# ENSEMBLE LEARNING
Combining Logistic Regression, Decision Tree, Random Forest, and XGBoost
using Voting, Stacking, and Bagging ensemble methods.

## 1. Setup & Data Preparation

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier,
    BaggingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from imblearn.over_sampling import SMOTE
from collections import Counter
import xgboost as xgb

print('All libraries loaded successfully!')

In [ ]:
train_v = pd.read_csv('./modified_diabetes_1.csv')

# Use the common feature set (decision tree feature set works for all models)
feature_set = [
    'age', 'time_in_hospital', 'num_procedures', 'num_medications',
    'number_outpatient_log', 'number_emergency_log', 'number_inpatient_log',
    'number_diagnoses', 'metformin', 'repaglinide', 'nateglinide',
    'chlorpropamide', 'glimepiride', 'glipizide', 'glyburide',
    'pioglitazone', 'rosiglitazone', 'acarbose', 'tolazamide', 'insulin',
    'glyburide-metformin', 'race_1', 'race_2', 'race_3', 'race_4', 'gender_1',
    'admission_source_id_4', 'admission_source_id_8',
    'admission_source_id_9', 'admission_source_id_11',
    'discharge_disposition_id_2', 'discharge_disposition_id_7',
    'discharge_disposition_id_10', 'discharge_disposition_id_18',
    'max_glu_serum_1.0', 'A1Cresult_1',
    'primary_diag_1', 'primary_diag_2', 'primary_diag_3', 'primary_diag_4',
    'primary_diag_5', 'primary_diag_6', 'primary_diag_7', 'primary_diag_8'
]

train_input  = train_v[feature_set]
train_output = train_v['readmitted']

print('Dataset shape:', train_v.shape)
print('Class distribution:', Counter(train_output))

In [ ]:
# Apply SMOTE for class imbalance
print('Original dataset shape: {}'.format(Counter(train_output)))
sm = SMOTE(random_state=20)
train_input_new, train_output_new = sm.fit_resample(train_input, train_output)
print('Resampled dataset shape: {}'.format(Counter(train_output_new)))

train_input_new = pd.DataFrame(train_input_new, columns=feature_set)

X_train, X_test, y_train, y_test = train_test_split(
    train_input_new, train_output_new, test_size=0.20, random_state=0
)
print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')

## 2. Define Base Models

In [ ]:
# Helper function to evaluate and display metrics
def evaluate_model(name, model, X_test, y_test, use_proba=True):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if use_proba else None
    
    metrics = {
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall'   : round(recall_score(y_test, y_pred), 4),
        'F1 Score' : round(f1_score(y_test, y_pred), 4),
        'ROC-AUC'  : round(roc_auc_score(y_test, y_proba if y_proba is not None else y_pred), 4)
    }
    return metrics, y_pred


def plot_confusion_matrix(name, y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{name} - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()


# Initialise base estimators (same hyperparams as your original models)
log_reg = LogisticRegression(fit_intercept=True, penalty='l2', random_state=0, max_iter=1000)

dec_tree = DecisionTreeClassifier(max_depth=28, criterion='entropy',
                                  min_samples_split=10, random_state=0)

rand_forest = RandomForestClassifier(n_estimators=10, max_depth=25,
                                     criterion='gini', min_samples_split=10,
                                     random_state=0)

xgb_model = xgb.XGBClassifier(
    booster='gbtree', objective='binary:logistic',
    eta=0.2, max_depth=6, colsample_bytree=0.7,
    n_estimators=200, use_label_encoder=False,
    eval_metric='logloss', random_state=0, verbosity=0
)

print('Base models defined.')

## 3. Train & Evaluate Individual Base Models

In [ ]:
results = []

for name, model in [
    ('Logistic Regression', log_reg),
    ('Decision Tree',       dec_tree),
    ('Random Forest',       rand_forest),
    ('XGBoost',             xgb_model)
]:
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    metrics, y_pred = evaluate_model(name, model, X_test, y_test)
    results.append(metrics)
    print(pd.DataFrame([metrics]).to_string(index=False))
    print('-' * 60)

base_results_df = pd.DataFrame(results)
print('\n=== BASE MODEL COMPARISON ===')
print(base_results_df.to_string(index=False))

## 4. Ensemble Method 1 — Hard & Soft Voting Classifier

In [ ]:
# --- Hard Voting (majority vote on class labels) ---
hard_voting = VotingClassifier(
    estimators=[
        ('lr',  LogisticRegression(fit_intercept=True, penalty='l2', random_state=0, max_iter=1000)),
        ('dt',  DecisionTreeClassifier(max_depth=28, criterion='entropy', min_samples_split=10, random_state=0)),
        ('rf',  RandomForestClassifier(n_estimators=10, max_depth=25, criterion='gini', min_samples_split=10, random_state=0)),
        ('xgb', xgb.XGBClassifier(booster='gbtree', objective='binary:logistic', eta=0.2,
                                   max_depth=6, colsample_bytree=0.7, n_estimators=200,
                                   use_label_encoder=False, eval_metric='logloss',
                                   random_state=0, verbosity=0))
    ],
    voting='hard'
)

print('Training Hard Voting Classifier...')
hard_voting.fit(X_train, y_train)
hv_metrics, hv_pred = evaluate_model('Hard Voting', hard_voting, X_test, y_test, use_proba=False)
hv_metrics['ROC-AUC'] = round(roc_auc_score(y_test, hv_pred), 4)   # no proba for hard voting
results.append(hv_metrics)
print(pd.DataFrame([hv_metrics]).to_string(index=False))
plot_confusion_matrix('Hard Voting', y_test, hv_pred)

In [ ]:
# --- Soft Voting (average predicted probabilities) ---
soft_voting = VotingClassifier(
    estimators=[
        ('lr',  LogisticRegression(fit_intercept=True, penalty='l2', random_state=0, max_iter=1000)),
        ('dt',  DecisionTreeClassifier(max_depth=28, criterion='entropy', min_samples_split=10, random_state=0)),
        ('rf',  RandomForestClassifier(n_estimators=10, max_depth=25, criterion='gini', min_samples_split=10, random_state=0)),
        ('xgb', xgb.XGBClassifier(booster='gbtree', objective='binary:logistic', eta=0.2,
                                   max_depth=6, colsample_bytree=0.7, n_estimators=200,
                                   use_label_encoder=False, eval_metric='logloss',
                                   random_state=0, verbosity=0))
    ],
    voting='soft'
)

print('Training Soft Voting Classifier...')
soft_voting.fit(X_train, y_train)
sv_metrics, sv_pred = evaluate_model('Soft Voting', soft_voting, X_test, y_test)
results.append(sv_metrics)
print(pd.DataFrame([sv_metrics]).to_string(index=False))
plot_confusion_matrix('Soft Voting', y_test, sv_pred)

## 5. Ensemble Method 2 — Stacking Classifier

In [ ]:
# Stacking: base learners feed predictions into a meta-learner (Logistic Regression)
stacking = StackingClassifier(
    estimators=[
        ('dt',  DecisionTreeClassifier(max_depth=28, criterion='entropy', min_samples_split=10, random_state=0)),
        ('rf',  RandomForestClassifier(n_estimators=10, max_depth=25, criterion='gini', min_samples_split=10, random_state=0)),
        ('xgb', xgb.XGBClassifier(booster='gbtree', objective='binary:logistic', eta=0.2,
                                   max_depth=6, colsample_bytree=0.7, n_estimators=200,
                                   use_label_encoder=False, eval_metric='logloss',
                                   random_state=0, verbosity=0))
    ],
    final_estimator=LogisticRegression(random_state=0, max_iter=1000),
    cv=5,
    passthrough=False    # set True to also pass raw features to meta-learner
)

print('Training Stacking Classifier (this may take a moment)...')
stacking.fit(X_train, y_train)
stack_metrics, stack_pred = evaluate_model('Stacking', stacking, X_test, y_test)
results.append(stack_metrics)
print(pd.DataFrame([stack_metrics]).to_string(index=False))
plot_confusion_matrix('Stacking', y_test, stack_pred)

## 6. Ensemble Method 3 — Bagging Classifier (on Decision Tree)

In [ ]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=28, criterion='entropy',
                                      min_samples_split=10, random_state=0),
    n_estimators=50,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    random_state=0,
    n_jobs=-1
)

print('Training Bagging Classifier...')
bagging.fit(X_train, y_train)
bag_metrics, bag_pred = evaluate_model('Bagging (DT)', bagging, X_test, y_test)
results.append(bag_metrics)
print(pd.DataFrame([bag_metrics]).to_string(index=False))
plot_confusion_matrix('Bagging (DT)', y_test, bag_pred)

## 7. Ensemble Method 4 — AdaBoost Classifier

In [ ]:
adaboost = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=4, random_state=0),
    n_estimators=200,
    learning_rate=0.1,
    random_state=0
)

print('Training AdaBoost Classifier...')
adaboost.fit(X_train, y_train)
ada_metrics, ada_pred = evaluate_model('AdaBoost', adaboost, X_test, y_test)
results.append(ada_metrics)
print(pd.DataFrame([ada_metrics]).to_string(index=False))
plot_confusion_matrix('AdaBoost', y_test, ada_pred)

## 8. Ensemble Method 5 — Gradient Boosting Classifier

In [ ]:
grad_boost = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    random_state=0
)

print('Training Gradient Boosting Classifier...')
grad_boost.fit(X_train, y_train)
gb_metrics, gb_pred = evaluate_model('Gradient Boosting', grad_boost, X_test, y_test)
results.append(gb_metrics)
print(pd.DataFrame([gb_metrics]).to_string(index=False))
plot_confusion_matrix('Gradient Boosting', y_test, gb_pred)

## 9. Cross-Validation Comparison of All Ensemble Methods

In [ ]:
cv_results = {}

ensemble_models = {
    'Hard Voting'      : hard_voting,
    'Soft Voting'      : soft_voting,
    'Stacking'         : stacking,
    'Bagging (DT)'     : bagging,
    'AdaBoost'         : adaboost,
    'Gradient Boosting': grad_boost
}

for name, model in ensemble_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:22s}  CV F1: {scores.mean():.4f} ± {scores.std():.4f}')

# Boxplot
plt.figure(figsize=(12, 6))
plt.boxplot(cv_results.values(), labels=cv_results.keys(), patch_artist=True)
plt.title('5-Fold Cross Validation F1 Score — Ensemble Methods')
plt.ylabel('F1 Score')
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 10. Final Comparison — All Models

In [ ]:
final_df = pd.DataFrame(results)
final_df = final_df.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print('=== FINAL MODEL COMPARISON (sorted by ROC-AUC) ===')
print(final_df.to_string(index=False))

In [ ]:
# Bar chart comparison
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
x = np.arange(len(final_df))
width = 0.15

fig, ax = plt.subplots(figsize=(16, 6))
for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i * width, final_df[metric], width, label=metric)

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('All Models — Performance Metrics Comparison')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(final_df['Model'], rotation=20, ha='right')
ax.set_ylim(0.5, 1.05)
ax.legend(loc='lower right')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of all metrics
heatmap_df = final_df.set_index('Model')[metrics_to_plot]
plt.figure(figsize=(10, 7))
sns.heatmap(heatmap_df, annot=True, fmt='.4f', cmap='YlGnBu',
            linewidths=0.5, vmin=0.5, vmax=1.0)
plt.title('Model Performance Heatmap')
plt.tight_layout()
plt.show()

print('\nBest model by ROC-AUC:', final_df.iloc[0]['Model'])